# Notebook 07: Handling Missing Data and Imbalance

## Why This Matters

Real-world datasets are never clean. Missing values are the norm in healthcare records, e-commerce data, and sensor streams. Class imbalance (99:1 fraud vs non-fraud) makes naive models useless. Handling these correctly separates production ML from notebook ML.

In [ ]:
%pip install -q scikit-learn imbalanced-learn pandas numpy matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from sklearn.model_selection import train_test_split, KFold
from sklearn.datasets import make_classification
from imblearn.over_sampling import SMOTE
import warnings; warnings.filterwarnings("ignore")
np.random.seed(42)
print("Ready.")

## 1. Types of Missingness

- **MCAR** (Missing Completely at Random): unrelated to any variable. Any imputation valid.
- **MAR** (Missing at Random): related to observed variables. MICE works well.
- **MNAR** (Missing Not at Random): related to the missing value itself. No unbiased imputation. Always add a missing indicator.

**Practical rule**: For MNAR features, always add a binary `feature_was_missing` flag. The fact that the value is missing IS itself a signal.

In [ ]:
n = 1000
age = np.random.randint(18, 80, n)
income = np.random.lognormal(10, 0.5, n)
target = (income > 60000).astype(int)
df = pd.DataFrame({"age": age, "income": income, "target": target})

df["income_mcar"] = df.income.copy(); df.loc[np.random.choice(n, 200), "income_mcar"] = np.nan
mar_prob = 0.05 + 0.2*(df.age > 50); df["income_mar"] = df.income.copy()
df.loc[np.random.binomial(1, mar_prob).astype(bool), "income_mar"] = np.nan
mnar_prob = 0.05 + 0.3*(df.income > 80000); df["income_mnar"] = df.income.copy()
df.loc[np.random.binomial(1, mnar_prob).astype(bool), "income_mnar"] = np.nan

print("=== Missingness Analysis ===")
for col in ["income_mcar", "income_mar", "income_mnar"]:
    missing = df[col].isna()
    corr = np.corrcoef(missing.astype(int), target)[0,1]
    print(f"{col}: missing={missing.mean():.1%}, corr_with_target={corr:.3f}")
print("\nMNAR: high corr -> missingness is informative -> add missing indicator!")

## 2. Imputation Strategies

- **Mean/median**: Fast, disrupts correlations. Baseline only.
- **KNN imputation**: Preserves correlations. O(n^2 * p) - expensive at scale.
- **MICE (IterativeImputer)**: Gold standard for MAR. Iteratively models each feature with all others.
- **Missing indicator**: Add binary flag BEFORE imputation. Always do this for MNAR features.

In [ ]:
X_miss = df[["age", "income_mnar"]].copy(); y_full = df.target
X_full = df[["age", "income"]].copy()
X_tr_full, X_te_full, y_tr, y_te = train_test_split(X_full, y_full, test_size=0.2, random_state=42)
X_tr_miss, X_te_miss = train_test_split(X_miss, test_size=0.2, random_state=42)[:2]

def eval_imp(X_tr, X_te, y_tr, y_te, imputer=None, add_indicator=False):
    Xtr = X_tr.values.copy(); Xte = X_te.values.copy()
    if add_indicator:
        Xtr = np.hstack([Xtr, np.isnan(Xtr[:,1]).reshape(-1,1).astype(float)])
        Xte = np.hstack([Xte, np.isnan(Xte[:,1]).reshape(-1,1).astype(float)])
    if imputer:
        Xtr[:,:2] = imputer.fit_transform(Xtr[:,:2]); Xte[:,:2] = imputer.transform(Xte[:,:2])
    sc = StandardScaler()
    return roc_auc_score(y_te, LogisticRegression(max_iter=500).fit(sc.fit_transform(Xtr), y_tr).predict_proba(sc.transform(Xte))[:,1])

sc = StandardScaler()
auc_t = roc_auc_score(y_te, LogisticRegression(max_iter=500).fit(sc.fit_transform(X_tr_full), y_tr).predict_proba(sc.transform(X_te_full))[:,1])
print(f"True (no missing): AUC = {auc_t:.4f}")
for name, imp, ind in [("Mean", SimpleImputer(), False), ("Mean+indicator", SimpleImputer(), True),
                        ("KNN", KNNImputer(n_neighbors=5), False), ("KNN+indicator", KNNImputer(n_neighbors=5), True)]:
    auc = eval_imp(X_tr_miss, X_te_miss, y_tr, y_te, imp, ind)
    print(f"{name:<20} AUC = {auc:.4f}", " <- indicator helps!" if ind else "")

## 3. Class Imbalance

Three strategies:
1. **class_weight balanced**: scales loss to weight minority class. Simplest - try first.
2. **SMOTE**: generates synthetic minority samples by interpolating between existing ones. Apply only to training data.
3. **Threshold tuning**: move decision threshold from 0.5 to optimize cost function.

In [ ]:
X, y = make_classification(n_samples=5000, n_features=20, weights=[0.98, 0.02], n_informative=5, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f"Training: {np.bincount(y_tr)} ({y_tr.mean():.2%} positive)")

lr_base = LogisticRegression(max_iter=500).fit(X_tr, y_tr)
lr_bal = LogisticRegression(class_weight="balanced", max_iter=500).fit(X_tr, y_tr)
X_sm, y_sm = SMOTE(random_state=42).fit_resample(X_tr, y_tr)
lr_smote = LogisticRegression(max_iter=500).fit(X_sm, y_sm)

print(f"\n=== Imbalanced Learning Comparison ===")
for name, probs in [("Baseline", lr_base.predict_proba(X_te)[:,1]), ("Balanced", lr_bal.predict_proba(X_te)[:,1]), ("SMOTE", lr_smote.predict_proba(X_te)[:,1])]:
    print(f"{name:<15} ROC-AUC={roc_auc_score(y_te,probs):.4f}  PR-AUC={average_precision_score(y_te,probs):.4f}")
print("\nPR-AUC is the key metric for imbalanced problems")

## 4. Threshold Optimization

In [ ]:
probs_bal = lr_bal.predict_proba(X_te)[:,1]
fn_cost, fp_cost = 500, 0.5
precision, recall, thresholds = precision_recall_curve(y_te, probs_bal)

best_cost, best_thresh = np.inf, 0.5
for thresh in thresholds:
    preds = (probs_bal >= thresh).astype(int)
    fn = ((preds==0)&(y_te==1)).sum(); fp = ((preds==1)&(y_te==0)).sum()
    cost = fn*fn_cost + fp*fp_cost
    if cost < best_cost: best_cost = cost; best_thresh = thresh

for label, thresh in [("Default (0.5)", 0.5), ("Cost-optimal", best_thresh)]:
    preds = (probs_bal >= thresh).astype(int)
    fn = ((preds==0)&(y_te==1)).sum(); fp = ((preds==1)&(y_te==0)).sum()
    cost = fn*fn_cost + fp*fp_cost
    print(f"{label} (t={thresh:.3f}): FN={fn}, FP={fp}, Cost=${cost:,.0f}")

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| MCAR/MAR/MNAR | MNAR: missingness is informative; always add missing indicator |
| Missing indicator | Binary flag for missing value; lets model learn missingness patterns |
| class_weight balanced | Simplest fix for imbalance; try this first |
| SMOTE | Synthetic minority samples; apply only to training data |
| Threshold tuning | Optimal threshold != 0.5; use cost matrix |
| PR-AUC | The right metric for imbalanced problems |

### Common Pitfalls
- Not adding a missing indicator for MNAR features
- Applying SMOTE before train/test split (leakage)
- Using accuracy for imbalanced data
- Setting threshold at 0.5 without considering business costs
